# Supervised Learning

## 1. Setup

Imports libraries and defines constants used throughout the notebook: the data directory, the four prediction checkpoints, and the rolling-average window for pregame features. The `read_data` helper loads CSVs from the shared `data/` folder and keeps `GAME_ID` as a string so its leading zeros are preserved across joins.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data")
CHECKPOINTS = ["_pre-game", "EndQ1", "EndQ2", "EndQ3"]
IN_GAME = ["EndQ1", "EndQ2", "EndQ3"]
ROLLING_WINDOW = 5  # default rolling average file to use
RANDOM_STATE = 42

def read_data(file_name):
    """Read a CSV from the data folder, casting GAME_ID as a string to keep leading zeros."""
    return pd.read_csv(DATA_DIR / f"{file_name}.csv", dtype = {"GAME_ID": str})

## 2. Load Data

Loads the CSVs produced by `nba_stats_data.ipynb` and `kalshi_data.ipynb`: full-game box scores (used to build leak-free rolling features, never as direct inputs), per-checkpoint game tables with Kalshi tickers and outcomes, in-game box stats at each quarter checkpoint, and Kalshi market snapshots. 

In [4]:
full = read_data("_team-box-stats(full-game)")
main = {checkpoint: read_data(f"main({checkpoint})")                    for checkpoint in CHECKPOINTS}
in_game = {checkpoint: read_data(f"in-game-box-stats({checkpoint})")    for checkpoint in IN_GAME}
kalshi = {checkpoint: read_data(f"kalshi({checkpoint})")                for checkpoint in CHECKPOINTS}
rolling_avg = read_data(f"roll-avg-only({ROLLING_WINDOW}GAME)")

**Sanity check: row counts and coverage.** Confirms each checkpoint table covers the expected number of games, how many games have a matching Kalshi market for each team, that the in-game and rolling stats have two team rows per game, and the home-team win rate that serves as the naive baseline.

In [12]:
for checkpoint in CHECKPOINTS:
    m, k = main[checkpoint], kalshi[checkpoint]
    tickers = set(k["KALSHI_ID"])
    print(
        f"{checkpoint:10s} games={m['GAME_ID'].nunique():5d}  "
        f"kalshi_rows={len(k):5d}  "  
        f"home_mkt_found={m['KALSHI_ID_HOME'].isin(tickers).mean():.1%}  "
        f"away_mkt_found={m['KALSHI_ID_AWAY'].isin(tickers).mean():.1%}"
    )
print()

for checkpoint in IN_GAME:
    print(f"in-game {checkpoint}: {len(in_game[checkpoint])} team rows -> {in_game[checkpoint]['GAME_ID'].nunique()} games")
print()

print(f"rolling ({ROLLING_WINDOW}g): {len(rolling_avg)} team rows -> {rolling_avg['GAME_ID'].nunique()} games")

home_win_rate = (main["_pre-game"]["WINNER"] == main["_pre-game"]["NAME_HOME"]).mean()
print(f"home win rate: {home_win_rate:.1%}")

_pre-game  games= 1225  kalshi_rows= 2446  home_mkt_found=99.8%  away_mkt_found=99.9%
EndQ1      games= 1225  kalshi_rows= 2446  home_mkt_found=99.8%  away_mkt_found=99.9%
EndQ2      games= 1225  kalshi_rows= 2443  home_mkt_found=99.6%  away_mkt_found=99.8%
EndQ3      games= 1225  kalshi_rows= 2432  home_mkt_found=98.9%  away_mkt_found=99.6%

in-game EndQ1: 2460 team rows -> 1230 games
in-game EndQ2: 2460 team rows -> 1230 games
in-game EndQ3: 2460 team rows -> 1230 games

rolling (5g): 2340 team rows -> 1173 games
home win rate: 55.4%


**Sanity check: games missing from the checkpoint tables.** Lists games present in the full-season box scores but absent from the Kalshi checkpoint tables. All five are neutral-site games (Mexico City, Las Vegas NBA Cup semifinals, Berlin, London), where home/away designation does not apply.

In [14]:
missing = set(full["GAME_ID"]) - set(main["_pre-game"]["GAME_ID"])

full[full["GAME_ID"].isin(missing)][["GAME_ID", "GAME_DATE", "MATCHUP"]].drop_duplicates("GAME_ID")

,GAME_ID,GAME_DATE,MATCHUP
166,0022500147,2025-11-01,DAL @ DET
744,0022501229,2025-12-13,NYK @ ORL
746,0022501230,2025-12-13,SAS @ OKC
1209,0022500578,2026-01-15,ORL @ MEM
1255,0022500602,2026-01-18,ORL @ MEM
